In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.src import data_loader

from algo_regime.src import regime_detector 




# (1) Data Pipeline Overview

## Index Universe

We track **9 major stock indices** across three regions:

| Region   | Index          | Ticker  |
|----------|----------------|---------|
| America  | S&P 500        | ^GSPC   |
| America  | Dow Jones      | ^DJI    |
| America  | Nasdaq         | ^IXIC   |
| Europe   | FTSE 100       | ^FTSE   |
| Europe   | CAC 40         | ^FCHI   |
| Europe   | DAX            | ^GDAXI  |
| Asia     | KOSPI          | ^KS11   |
| Asia     | Nikkei 225     | ^N225   |
| Asia     | Hang Seng      | ^HSI    |

## Raw Data Structure

Each index is downloaded from Yahoo Finance with **daily OHLCV** columns:
```
Date | Open | High | Low | Close | Volume
```

After merging, the full DataFrame uses a **MultiIndex on columns**: `(index_name, OHLCV_field)`, so accessing S&P 500 closing prices looks like `df["SP500"]["Close"]`.

## Feature Engineering

From the closing prices, two families of features are computed:

**1. Log-returns** over multiple horizons (default: 1, 5, 21 trading days):

$$r_{t,p} = \ln\left(\frac{P_t}{P_{t-p}}\right)$$

**2. Rolling annualised volatility** (default windows: 21 and 63 days):

$$\sigma_{t,w} = \text{std}\left(r_{t-w:t}\right) \times \sqrt{252}$$

This produces a feature matrix of shape `(T, 9 × (3 returns + 2 vols)) = (T, 45)` columns, where each row is one trading day and each column is `(index_name, feature_name)`.

## Pipeline Summary
```
Yahoo Finance API
       │
       ▼
  Raw OHLCV (per index)
       │
       ▼
  Merged DataFrame (MultiIndex columns)
       │
       ▼
  Close prices only (simple DataFrame)
       │
       ├──► Log-returns (1d, 5d, 21d)
       │
       ├──► Rolling volatility (21d, 63d)
       │
       ▼
  Feature matrix → Clustering / PCA
```

In [ ]:
start_date = '2006-01-01'
end_date = '2020-01-01'
interval = '1d'



## (1.1) America Data 

- Dow Jones Index
- S&P 500 Index
- NASDAQ Index 

In [ ]:
df_america = data_loader.get_close_prices(start_date=start_date, end_date=end_date, regions=['america'], interval=interval)
df_america.head()
print(df_america.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=df_america.drop(columns=['SP500', 'Nasdaq']), ax=ax)
ax2 = ax.twinx()
sns.lineplot(data=df_america[['SP500', 'Nasdaq']], ax=ax2, palette=['red', 'orangered'])
ax2.set_ylabel('Index Close Price for S&P500 and Nasdaq', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax.set_title('American Indices Close Prices')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (Dow Jones)', color= 'b')
ax.tick_params(axis='y', labelcolor='b')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()

# (1.2) Asia Data 

- Nikkei 225
- Kospi 
- Hang Seng 

In [ ]:
df_asia = data_loader.get_close_prices(start_date=start_date, end_date=end_date, regions=['asia'], interval=interval)
df_asia.head()

In [ ]:
fig ,ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=df_asia.drop(columns=['KOSPI']), ax=ax)
ax.set_title('Asian Indices Close Prices')
#add a  different scale for the KOSPI index
ax2 = ax.twinx()
sns.lineplot(data=df_asia['KOSPI'], ax=ax2, color='r')
ax2.set_ylabel('KOSPI Close Price', color='r')
ax2.tick_params(axis='y', labelcolor='r')


ax.set_xlabel('Date')
ax.set_ylabel('Nikkei and Hang Seng Close Prices')
plt.show()

# (1.3) Europe Data 

- Cac 40
- DAX
- FTSE 100

In [ ]:
df_europe = data_loader.get_close_prices(start_date=start_date, end_date=end_date, regions=['europe'], interval=interval)
df_europe.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_europe, ax=plt.gca())
plt.title('European Indices Close Prices')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.show()

# (2) K-means with PCA to detect regime  

## The Mathematics of K-Means Regime Detection

### Phase 1: Feature Engineering (State Encoding)

To transform sequential time-series data into cross-sectional features, we calculate several rolling metrics for each trading day $t$:

**1. Log Returns**
Ensures stationarity and additive symmetry over $p$ periods:
$$r_{t, p} = \ln\left(\frac{P_t}{P_{t-p}}\right)$$

**2. Annualized Rolling Volatility**
Measures dispersion over a rolling window $w$ (scaled to 252 trading days):
$$\sigma_{t, w} = \sqrt{252} \cdot \sqrt{\frac{1}{w-1} \sum_{i=0}^{w-1} \left(r_{t-i} - \mu_{w}\right)^2}$$

**3. Rolling Cross-Correlation (Pearson)**
Captures co-movement between indices $X$ and $Y$, which often spikes during regime shifts (e.g., market panics):
$$\rho_{X,Y, t} = \frac{\sum_{i=0}^{w-1} (r_{X, t-i} - \bar{r}_X)(r_{Y, t-i} - \bar{r}_Y)}{\sqrt{\sum_{i=0}^{w-1} (r_{X, t-i} - \bar{r}_X)^2 \sum_{i=0}^{w-1} (r_{Y, t-i} - \bar{r}_Y)^2}}$$

**4. Momentum Z-Score**
Measures mean-reversion by normalizing the distance between current price and a moving average:
$$Z_t = \frac{P_t - \mu_{\text{MA}}}{\sigma_{\text{lookback}}}$$

---

### Phase 2: Dimensionality Reduction

**1. Standardization**
Normalizes features so algorithms based on Euclidean distance do not overweight features with larger absolute bounds:
$$z_{t,j} = \frac{x_{t,j} - \mu_j}{\sigma_j}$$

**2. Principal Component Analysis (PCA)**
Addresses collinearity by projecting the matrix onto an orthogonal subspace. It solves for the eigenvectors $\mathbf{v}$ and eigenvalues $\lambda$ of the covariance matrix $\boldsymbol{\Sigma}$:
$$\boldsymbol{\Sigma} \mathbf{v} = \lambda \mathbf{v}$$

---

### Phase 3: K-Means Objective Function

K-Means seeks to partition the trading days into $K$ distinct regimes by minimizing the **Within-Cluster Sum of Squares (WCSS)**, or Inertia $J$. It minimizes the squared distance between each day's feature vector $\mathbf{x}_t$ and its assigned regime centroid $\boldsymbol{\mu}_k$:
$$J = \sum_{k=1}^{K} \sum_{\mathbf{x}_t \in S_k} \|\mathbf{x}_t - \boldsymbol{\mu}_k\|^2$$

---

### Phase 4: Optimal $K$-Selection

To rigorously determine the correct number of regimes, we calculate the **Silhouette Score** for each day $i$. This metric evaluates cohesion $a(i)$ (average distance to days in the same regime) versus separation $b(i)$ (average distance to days in the nearest different regime):
$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$
*A mean score closer to 1.0 indicates dense, well-separated regimes.*

## (2.1) Regime detection on American Indices 

In [ ]:
from algo_regime.src import regime_detector
from algo_regime.src import metrics 
import importlib
importlib.reload(regime_detector)
importlib.reload(metrics)
rd_america = regime_detector.RegimeDetector(df_america, use_pca = True, n_regimes= None)

rd_america.fit()
regime_labels_ptt = rd_america.peak_to_trough_labelling(drawdown_threshold=-0.1)
skmeans_labels = rd_america.detect_regime_skmeans(N_S = 5, L = 100, h1 = 50, h2 = 10, epsilon = 1e-6)

In [ ]:

rd_america.generate_signals()
bt_america = rd_america.backtest(initial_capital=100)

fig = rd_america.plot_regime_profile()

In [ ]:
rd_america.plot_k_selection()

In [ ]:
fig = rd_america.plot_cluster_pca()

In [ ]:
#Plot balanced accuracy score of the regime labels against the peak-to-trough labels
from sklearn.metrics import balanced_accuracy_score
balanced_acc = balanced_accuracy_score(rd_america.labels, rd_america.regime_labels_ptt)
print(f"Balanced Accuracy Score of Regime Labels vs Peak-to-Trough Labels: {balanced_acc:.4f}")

In [ ]:
bt_america = rd_america.backtest(initial_capital=100)
fig = rd_america.plot_backtest(bt_america)

In [ ]:
bt_america_skmeans = rd_america.backtest_skmeans(initial_capital=100)
print(bt_america_skmeans.head())
plt.plot(df_america.index, bt_america_skmeans['cum_strategy_skmeans'], label='sWkmeans Strategy')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.title('Cumulative Return of sWkmeans Strategy')
plt.legend()
plt.show()


In [ ]:
print("-- Backtest Summary --")
print('---------------------- Strategy Performance ----------------------')
print(f"Final Cumulative Return : {bt_america['cum_strategy'].iloc[-1]:.2f}")
print(f"Sharpe Ratio : {bt_america['sharpe_strategy'].iloc[-1]:.2f}")

print("---------------------- Peak-to-Trough Strategy Performance ----------------------")
print(f"Final Cumulative Return (PTT): {bt_america['cum_strategy_ptt'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (PTT): {bt_america['sharpe_strategy_ptt'].iloc[-1]:.2f}")

print('---------------------- sWkmeans Strategy Performance ----------------------')
print(f"Final Cumulative Return (sWkmeans): {bt_america_skmeans['cum_strategy_skmeans'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (sWkmeans): {bt_america_skmeans['sharpe_strategy_skmeans'].iloc[-1]:.2f}")

print('---------------------- Long Only Performance ----------------------')
print(f"Final Cumulative Return (Basket): {bt_america['cum_basket'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (Basket): {bt_america['sharpe_basket'].iloc[-1]:.2f}") 



## (2.2) Regime Detection on European Indices 

In [ ]:
from algo_regime.src import regime_detector 
import importlib
importlib.reload(regime_detector)
rd_europe = regime_detector.RegimeDetector(df_europe, use_pca = True, n_regimes= None)
rd_europe.fit()
regime_labels_ptt = rd_europe.peak_to_trough_labelling(drawdown_threshold=-0.1)



### Elbow Plot

In [ ]:
rd_europe.plot_k_selection()

### Regime Prediction 

In [ ]:
fig = rd_europe.plot_regime_profile()

In [ ]:
fig = rd_europe.plot_cluster_pca()

## Plot Backtest 

In [ ]:
from algo_regime.src import regime_detector 
importlib.reload(regime_detector)

bt = rd_europe.backtest(initial_capital=100)
fig = rd_europe.plot_backtest()

In [ ]:
print("-- Backtest Summary --")
print('---------------------- Strategy Performance ----------------------')
print(f"Final Cumulative Return : {bt['cum_strategy'].iloc[-1]:.2f}")
print(f"Sharpe Ratio : {bt['sharpe_strategy'].iloc[-1]:.2f}")

print("---------------------- Peak-to-Trough Strategy Performance ----------------------")
print(f"Final Cumulative Return (PTT): {bt['cum_strategy_ptt'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (PTT): {bt['sharpe_strategy_ptt'].iloc[-1]:.2f}")  

print('---------------------- Long Only Performance ----------------------')
print(f"Final Cumulative Return (Basket): {bt['cum_basket'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (Basket): {bt['sharpe_basket'].iloc[-1]:.2f}") 



## (2.3) Regime detection on Asian Indices 

In [ ]:
rd_asia = regime_detector.RegimeDetector(df_asia, use_pca = True, n_regimes= None)

rd_asia.fit()
rd_asia.peak_to_trough_labelling(drawdown_threshold=-0.1)
rd_asia.generate_signals()
rd_asia.backtest()


In [ ]:
rd_asia.plot_k_selection()

In [ ]:
fig =rd_asia.plot_regime_profile()

In [ ]:
fig = rd_asia.plot_cluster_pca()

In [ ]:
bt_asia = rd_asia.backtest(initial_capital=100)
fig = rd_asia.plot_backtest()

In [ ]:
print("-- Backtest Summary --")
print('---------------------- Strategy Performance ----------------------')
print(f"Final Cumulative Return : {bt_asia['cum_strategy'].iloc[-1]:.2f}")
print(f"Sharpe Ratio : {bt_asia['sharpe_strategy'].iloc[-1]:.2f}")

print("---------------------- Peak-to-Trough Strategy Performance ----------------------")
print(f"Final Cumulative Return (PTT): {bt_asia['cum_strategy_ptt'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (PTT): {bt_asia['sharpe_strategy_ptt'].iloc[-1]:.2f}")

print('---------------------- Long Only Performance ----------------------')
print(f"Final Cumulative Return (Basket): {bt_asia['cum_basket'].iloc[-1]:.2f}")
print(f"Sharpe Ratio (Basket): {bt_asia['sharpe_basket'].iloc[-1]:.2f}") 



# (3) Hyperparameter tuning for the K-Mean Regime Detection Algorithm 

# (4) Model Comparison with Guassian Mixture Models

# TODO 